[Link to Homework Questions](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework1.md)

## Question 1: [Index] S&P 500 Stocks Added to the Index

### Which year had the highest number of additions (starting from 2020)?

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime as dt


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# send request to url
response = requests.get(url, headers=headers)
response.raise_for_status()

# get table
soup = BeautifulSoup(response.content, "html.parser")
table = soup.find("table", id="constituents")

df = pd.read_html(str(table))[0]

# Extract year added from Date added
df["Date added"] = pd.to_datetime(df["Date added"])
df["year_added"] = df["Date added"].dt.year

# group by year added and count companies added in that year
df_yearly_add = df[["year_added", "Symbol"]].groupby(
    'year_added'
  ).count().reset_index()

# filter to 2020 and above only
df_filtered = df_yearly_add[df_yearly_add.year_added >= 2020]
display(df_filtered)
print(df_filtered[df_filtered.Symbol == max(df_filtered.Symbol)])

/tmp/ipykernel_8414/1060976406.py:20: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


,year_added,Symbol
51,2020,10
52,2021,10
53,2022,15
54,2023,15
55,2024,16
56,2025,18
57,2026,13


    year_added  Symbol
56        2025      18


Answer: From 2020 onwards, The year with the most companies added is 2025 with 18 additions.

### Additional: How many current S&P 500 stocks have been in the index for more than 20 years?

In [ ]:
df_old_companies = df[df.year_added < dt.today().year - 20]
display(df_old_companies.head())
display(df_old_companies.shape[0])

### double check:
# ((dt.now().year - df.year_added) > 20).sum()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,year_added
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,1957
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,1957
5,ADBE,Adobe Inc.,Information Technology,Application Software,"San Jose, California",1997-05-05,796343,1982,1997
7,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders,"Arlington, Virginia",1998-10-02,874761,1981,1998
8,AFL,Aflac,Financials,Life & Health Insurance,"Columbus, Georgia",1999-05-28,4977,1955,1999


218

Answer: There are 218 companies in the S&P500 that has been there for at least 20 years.

## Question 2. [Macro] Indexes YTD (as of 21 August 2026)

### How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?

In [ ]:
import datetime as dt
import yfinance as yf
from time import sleep

# build indices dict to loop through
indices = [
    {"index": "^GSPC", "country": "US"},
    {"index": "000001.SS", "country": "China"},
    {"index": "^HSI", "country": "Hong Konf"},
    {"index": "^AXJO", "country": "Australia"},
    {"index": "^NSEI", "country": "India"},
    {"index": "^GSPTSE", "country": "Canada"},
    {"index": "^GDAXI", "country": "Germany"},
    {"index": "^FTSE", "country": "UK"},
    {"index": "^N225", "country": "Japan"},
    {"index": "^MXX", "country": "Mexico"},
    {"index": "^BVSP", "country": "Brazil"}
]
start = dt.date(year=2026, month=1, day=1)
end = dt.date(year=2026, month=8, day=21)

# get data then calculate YTD growth and save to dict
for i in indices:
  print(f"Getting data for {i["country"]}.")
  tempdf = yf.Ticker(i["index"]).history(start=start, end=end)
  tempdf['ytd_growth'] = tempdf.Close / tempdf.Close.shift(
    tempdf.shape[0] - 1
  ) - 1
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]
  sleep(2)

# convert dict to DataFrame
df = pd.DataFrame(indices)

# find countries with better YTD growth than the US
display(df[df.ytd_growth > df[df.country == "US"].ytd_growth.iloc[0]])

Getting data for US.


/tmp/ipykernel_8414/4261642315.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]


Getting data for China.


/tmp/ipykernel_8414/4261642315.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]


Getting data for Hong Konf.


...

Getting data for Mexico.


/tmp/ipykernel_8414/4261642315.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]


Getting data for Brazil.


/tmp/ipykernel_8414/4261642315.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]


,index,country,ytd_growth
5,^GSPTSE,Canada,0.140575
8,^N225,Japan,0.277507


Answer: There are 2 countries performing better, whihc are Canada and Japan.

## Question 3. [Index] S&P 500 Market Corrections Analysis

### Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.

In [ ]:
import yfinance as yf
import datetime as dt
import pandas as pd

In [ ]:
# set start and end date parameters
start = dt.date(year=1950, month=1, day=1)
end = dt.date.today()

# retrieve data
dfraw = yf.Ticker("^GSPC").history(start=start, end=end)

In [ ]:
# trim columns
df = dfraw.reset_index()[['Date', 'Close']]

# # convert to date type (daily grain)
# df['Date'] = df['Date'].dt.date

# Calculate the all-time high closing price up to each day
df['all_time_high_close'] = df['Close'].expanding().max()

# Identify days where the Close price is an all-time high
df['is_all_time_high'] = (df['Close'] == df['all_time_high_close'])

df["consecutive_ath"] = df.is_all_time_high & df.is_all_time_high.shift(1)
# df["prev_ath"] = df.all_time_high_close.shift(1)

display(df.head(3))

,Date,Close,all_time_high_close,is_all_time_high,consecutive_ath
0,1950-01-03 00:00:00-05:00,16.66,16.66,True,False
1,1950-01-04 00:00:00-05:00,16.85,16.85,True,True
2,1950-01-05 00:00:00-05:00,16.93,16.93,True,True
3,1950-01-06 00:00:00-05:00,16.98,16.98,True,True
4,1950-01-09 00:00:00-05:00,17.08,17.08,True,True


In [ ]:
def get_date_of_min_close(group_close_series):
    # find the index of min in the groupby subset from original df and return the date
    return df.loc[group_close_series.idxmin(), 'Date']

# aggregate by all time high closing value
df_ath_agg = df[['Date', 'Close', 'all_time_high_close']].groupby('all_time_high_close').agg(
    first_date=('Date', 'min'),
    last_date=('Date', 'max'),
    num_days=('Date', 'count'),
    min_close=('Close', 'min'),
    min_close_date=('Close', get_date_of_min_close)
).reset_index()

# compute drawdown percentage
df_ath_agg['drawdown_perc'] = (
    (df_ath_agg['all_time_high_close'] - df_ath_agg['min_close']) / df_ath_agg['all_time_high_close']
) * 100
# alt: ((df_ath_agg['all_time_high_close'] / df_ath_agg['min_close']) - 1) * 100

# get duration of drawdown in days
df_ath_agg['correction_duration'] = (df_ath_agg['min_close_date'] - df_ath_agg['first_date']).dt.days

display(df_ath_agg.head(3))

,all_time_high_close,first_date,last_date,num_days,min_close,min_close_date,drawdown_perc,correction_duration
0,16.66,1950-01-03 00:00:00-05:00,1950-01-03 00:00:00-05:00,1,16.660000,1950-01-03 00:00:00-05:00,0.000000,0
1,16.85,1950-01-04 00:00:00-05:00,1950-01-04 00:00:00-05:00,1,16.850000,1950-01-04 00:00:00-05:00,0.000000,0
2,16.93,1950-01-05 00:00:00-05:00,1950-01-05 00:00:00-05:00,1,16.930000,1950-01-05 00:00:00-05:00,0.000000,0
3,16.98,1950-01-06 00:00:00-05:00,1950-01-06 00:00:00-05:00,1,16.980000,1950-01-06 00:00:00-05:00,0.000000,0
4,17.08,1950-01-09 00:00:00-05:00,1950-01-10 00:00:00-05:00,2,17.030001,1950-01-10 00:00:00-05:00,0.292736,1


In [ ]:
# quick sanity check
df_ath_agg.sort_values(by='drawdown_perc', ascending=False).head(10)

,all_time_high_close,first_date,last_date,num_days,min_close,min_close_date,drawdown_perc,correction_duration
1040,1565.150024,2007-10-09 00:00:00-04:00,2013-03-27 00:00:00-04:00,1376,676.530029,2009-03-09 00:00:00-04:00,56.775388,517
1031,1527.459961,2000-03-24 00:00:00-05:00,2007-05-29 00:00:00-04:00,1803,776.760010,2002-10-09 00:00:00-04:00,49.146948,928
528,120.239998,1973-01-11 00:00:00-05:00,1980-07-16 00:00:00-04:00,1898,62.279999,1974-10-03 00:00:00-04:00,48.203593,629
493,108.370003,1968-11-29 00:00:00-05:00,1972-03-03 00:00:00-05:00,820,69.290001,1970-05-26 00:00:00-04:00,36.061641,542
1295,3386.149902,2020-02-19 00:00:00-05:00,2020-08-17 00:00:00-04:00,126,2237.399902,2020-03-23 00:00:00-04:00,33.924960,32
704,336.769989,1987-08-25 00:00:00-04:00,1989-07-25 00:00:00-04:00,485,223.919998,1987-12-04 00:00:00-05:00,33.509515,101
327,72.639999,1961-12-12 00:00:00-05:00,1963-08-30 00:00:00-04:00,434,52.320000,1962-06-26 00:00:00-04:00,27.973568,195
552,140.520004,1980-11-28 00:00:00-05:00,1982-11-02 00:00:00-05:00,488,102.419998,1982-08-12 00:00:00-04:00,27.113582,621
1386,4796.560059,2022-01-03 00:00:00-05:00,2024-01-18 00:00:00-05:00,513,3577.030029,2022-10-12 00:00:00-04:00,25.425097,281
446,94.059998,1966-02-09 00:00:00-05:00,1967-05-03 00:00:00-04:00,310,73.199997,1966-10-07 00:00:00-04:00,22.177335,239


In [ ]:
# filter to at least 5% drawdown
df_filtered = df_ath_agg[df_ath_agg.drawdown_perc >= 5.0]

# calculate percentiles for correction_duration
print("Correction Duration Percentiles (Days):")
print(df_filtered['correction_duration'].quantile([0.25, 0.5, 0.75]))

# calculate percentiles for drawdown percentage
print("\nDrawdown Percentage Percentiles:")
print(df_filtered['drawdown_perc'].quantile([0.25, 0.5, 0.75]))

Correction Duration Percentiles (Days):
0.25    22.0
0.50    40.5
0.75    86.0
Name: correction_duration, dtype: float64

Drawdown Percentage Percentiles:
0.25     6.234677
0.50     7.986358
0.75    14.019826
Name: drawdown_perc, dtype: float64


Answer: The median drawdown for significant market corrections is 7.98% or roughly 8%.

## Question 4. [Stocks] Earnings Surprise Analysis for Amazon (AMZN)

### Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.

In [65]:
import yfinance as yf
import pandas as pd
import datetime as dt


In [66]:
# get earnings data
ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
result = ticker_obj.get_earnings_dates()

display(result.shape)
display(result.tail(3))

(25, 3)

,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2021-04-29 16:00:00-04:00,0.47,0.79,67.30
2021-02-02 16:00:00-05:00,0.35,0.70,100.10
2020-10-29 16:00:00-04:00,0.38,0.62,64.25


In [67]:
# reset index to get earnings date and rename columns
df_earnings = result.reset_index()
df_earnings.rename(inplace=True,
                   columns={
    "Earnings Date": "earnings_date",
    "EPS Estimate": "eps_estimate",
    "Reported EPS": "reported_eps",
    "Surprise(%)": "perc_surprise"
    },
)

# use string date to join later with stock OHLCV table
df_earnings["earnings_date_str"] = df_earnings.earnings_date.dt.strftime('%Y-%m-%d')

# filter for positive surprises only
df_earnings["is_positive"] = df_earnings.perc_surprise >= 0
df_earnings = df_earnings[df_earnings.is_positive]

display(df_earnings.head())

,earnings_date,eps_estimate,reported_eps,perc_surprise,earnings_date_str,is_positive
1,2026-07-30 16:00:00-04:00,1.83,5.75,215.02,2026-07-30,True
2,2026-04-29 16:00:00-04:00,1.64,2.78,69.02,2026-04-29,True
3,2026-02-05 16:00:00-05:00,1.95,1.95,0.22,2026-02-05,True
4,2025-10-30 16:00:00-04:00,1.56,1.95,25.20,2025-10-30,True
5,2025-07-31 16:00:00-04:00,1.32,1.68,27.19,2025-07-31,True


In [68]:
# retrieve dataset min and max dates (adjust 1 extra day before and after)
start = (df_earnings.earnings_date.min() - dt.timedelta(days=1)).strftime("%Y-%m-%d")
end = (df_earnings.earnings_date.max() + dt.timedelta(days=1)).strftime("%Y-%m-%d")

# get stock ticker date within the date range covered
ticker_ohlcv = ticker_obj.history(start=start, end=end)

print(f"Start Date: {start} \nEnd Date: {end}")
display(ticker_ohlcv.shape)

Start Date: 2020-10-28 
End Date: 2026-07-31


(1444, 7)

In [69]:
df = ticker_ohlcv.reset_index()[['Date', 'Close']].rename(columns={'Date': 'date', 'Close': 'close'})

# use to join with earnings table
df['date_str'] = df.date.dt.strftime('%Y-%m-%d')

# calculate 1 day before and 1 day after close and their percentage diff
df["close_day1"] = df.close.shift(1)
df["close_day3"] = df.close.shift(-1)
df["day3_day1_perc_change"] = ((df.close_day3 / df.close_day1) - 1) * 100

df.head()

,date,close,date_str,close_day1,close_day3,day3_day1_perc_change
0,2020-10-28 00:00:00-04:00,158.139008,2020-10-28,NaN,160.550507,NaN
1,2020-10-29 00:00:00-04:00,160.550507,2020-10-29,158.139008,151.807495,-4.003764
2,2020-10-30 00:00:00-04:00,151.807495,2020-10-30,160.550507,150.223999,-6.431937
3,2020-11-02 00:00:00-05:00,150.223999,2020-11-02,151.807495,152.420502,0.403805
4,2020-11-03 00:00:00-05:00,152.420502,2020-11-03,150.223999,162.057999,7.877569


In [70]:
# filter for only positive earnings surprises using inner join
df_filtered = pd.merge(df,
                       df_earnings,
                       left_on='date_str',
                       right_on='earnings_date_str',
                       how='inner')

display(df_filtered.shape)
display(df_filtered.head())

(20, 12)

,date,close,date_str,close_day1,close_day3,day3_day1_perc_change,earnings_date,eps_estimate,reported_eps,perc_surprise,earnings_date_str,is_positive
0,2020-10-29 00:00:00-04:00,160.550507,2020-10-29,158.139008,151.807495,-4.003764,2020-10-29 16:00:00-04:00,0.38,0.62,64.25,2020-10-29,True
1,2021-02-02 00:00:00-05:00,169.000000,2021-02-02,167.143997,165.626495,-0.907901,2021-02-02 16:00:00-05:00,0.35,0.70,100.10,2021-02-02,True
2,2021-04-29 00:00:00-04:00,173.565506,2021-04-29,172.925003,173.371002,0.257915,2021-04-29 16:00:00-04:00,0.47,0.79,67.30,2021-04-29,True
3,2021-07-29 00:00:00-04:00,179.996002,2021-07-29,181.516006,166.379501,-8.338937,2021-07-29 16:00:00-04:00,0.61,0.76,23.06,2021-07-29,True
4,2022-02-03 00:00:00-05:00,138.845505,2022-02-03,150.612503,157.639496,4.665611,2022-02-03 16:00:00-05:00,0.18,1.39,657.12,2022-02-03,True


In [71]:
median_2day_return = df_filtered['day3_day1_perc_change'].median()
correlation = df_filtered["day3_day1_perc_change"].corr(df_filtered['perc_surprise'])

print(f"Answer: The median 2-day percentage change following a positive earnings surprise is: {median_2day_return:.2f}%")

Answer: The median 2-day percentage change following a positive earnings surprise is: 0.26%


Additional: Is there a correlation between the magnitude of the earnings surprise and the stock price reaction? Does the market react differently to earnings surprises during bull vs. bear markets?

In [72]:
print(f"Answer: The correlation between 2-day percentage change and Surprise (%) is {correlation:.2f} which suggest there is very low positive correlation.")

Answer: The correlation between 2-day percentage change and Surprise (%) is 0.25 which suggest there is very low positive correlation.


## Question 5. [Exploratory, optional] Brainstorm potential idea for your capstone project

Describe the capstone project you would like to pursue, considering your aspirations, ML model predictions, and prior knowledge. Even if you are unsure at this stage, try to generate an idea you would like to explore-such as a specific asset class, country, industry vertical, or investment strategy. Be as specific as possible.

Example: I want to build a short-term prediction model for the US/India/Brazil stock markets, focusing on the largest stocks over a 30-day investment horizon. I plan to use RSI and MACD technical indicators and news coverage data to generate predictions.

Answer: Capture short-to-medium-term post-earnings announcement drift (PEAD) or mean reversion in high-liquidity US tech stocks (e.g., Apple, Microsoft, Nvidia) over a 5-day to 15-day investment horizon.

## Question 6. [Exploratory, optional] Investigate new metrics

Using the data sources we have covered (or any others you find relevant), download and explore a few additional metrics or time series that could be valuable for your project. Briefly explain why you think each metric is useful. This does not need to be a comprehensive list-focus on demonstrating your ability to generate data requests based on your project description, identify and locate the necessary data, and explain how you would retrieve it using Python.